## **TASK 1**

In [1]:
!pip install pandas spacy sentence-transformers faiss-cpu requests
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 40.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 70.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
# --- Imports ---
import requests
import pandas as pd
import time
import json
from datetime import datetime, timezone

# --- Function to fetch posts, with retry protection ---
def fetch_posts(subreddit, after, before, limit=100, max_retries=5):
    url = "https://arctic-shift.photon-reddit.com/api/posts/search"
    all_posts = []
    params = {"subreddit": subreddit, "after": after, "before": before, "limit": limit, "sort": "asc"}

    while True:
        retries = 0
        response = None
        while retries < max_retries:
            try:
                response = requests.get(url, params=params, timeout=30)
                if response.status_code == 200:
                    break
                print(f"Status {response.status_code}, retrying... ({retries+1}/{max_retries})")
            except requests.exceptions.RequestException as e:
                print(f"Connection error, retrying... ({e})")
                response = None
            time.sleep(5)
            retries += 1

        if response is None or response.status_code != 200:
            print("Stopping, saving what we have.")
            break

        data = response.json()
        posts = data.get("data", [])
        if not posts:
            print("No more posts — done.")
            break

        all_posts.extend(posts)
        print(f"Posts collected: {len(all_posts)}")

        # Save progress after every batch so nothing is ever lost
        with open("posts_progress.json", "w") as f:
            json.dump(all_posts, f)

        last_created = posts[-1]["created_utc"]
        next_date = datetime.fromtimestamp(last_created, tz=timezone.utc)
        params["after"] = next_date.strftime("%Y-%m-%dT%H:%M:%S")
        time.sleep(2.5)

        if len(posts) < limit:
            break

    return all_posts

# --- Run it ---
raw_posts = fetch_posts("artificial", "2026-06-01", "2026-06-30")
posts_df = pd.DataFrame(raw_posts)

print(f"\nTOTAL POSTS: {len(raw_posts)}")
print("Posts shape:", posts_df.shape)

Status 422, retrying... (1/5)
Posts collected: 100
Posts collected: 200
Posts collected: 300
Status 422, retrying... (1/5)
Posts collected: 400
Status 422, retrying... (1/5)
Posts collected: 500
Status 422, retrying... (1/5)
Status 422, retrying... (2/5)
Posts collected: 600
Posts collected: 700
Status 422, retrying... (1/5)
Posts collected: 800
Posts collected: 900
Status 422, retrying... (1/5)
Posts collected: 1000
Status 422, retrying... (1/5)
Status 422, retrying... (2/5)
Posts collected: 1100
Posts collected: 1200
Status 422, retrying... (1/5)
Posts collected: 1300
Posts collected: 1400
Status 422, retrying... (1/5)
Posts collected: 1500
Posts collected: 1600
Status 422, retrying... (1/5)
Posts collected: 1700
Posts collected: 1800
Posts collected: 1900
Status 422, retrying... (1/5)
Posts collected: 2000
Status 422, retrying... (1/5)
Status 422, retrying... (2/5)
Posts collected: 2100
Status 422, retrying... (1/5)
Posts collected: 2200
Status 422, retrying... (1/5)
Posts collected

In [8]:

# --- Function to fetch comments, with retry protection ---
def fetch_comments(subreddit, after, before, limit=100, max_retries=5):
    url = "https://arctic-shift.photon-reddit.com/api/comments/search"
    all_comments = []
    params = {"subreddit": subreddit, "after": after, "before": before, "limit": limit, "sort": "asc"}

    while True:
        retries = 0
        response = None
        while retries < max_retries:
            try:
                response = requests.get(url, params=params, timeout=30)
                if response.status_code == 200:
                    break
                print(f"Status {response.status_code}, retrying... ({retries+1}/{max_retries})")
            except requests.exceptions.RequestException as e:
                print(f"Connection error, retrying... ({e})")
                response = None
            time.sleep(5)
            retries += 1

        if response is None or response.status_code != 200:
            print("Stopping, saving what we have.")
            break

        data = response.json()
        comments = data.get("data", [])
        if not comments:
            print("No more comments — done.")
            break

        all_comments.extend(comments)
        print(f"Comments collected: {len(all_comments)}")

        # Save progress after every batch so nothing is ever lost
        with open("comments_progress.json", "w") as f:
            json.dump(all_comments, f)

        last_created = comments[-1]["created_utc"]
        next_date = datetime.fromtimestamp(last_created, tz=timezone.utc)
        params["after"] = next_date.strftime("%Y-%m-%dT%H:%M:%S")
        time.sleep(2.5)

        if len(comments) < limit:
            break

    return all_comments

# --- Run it ---
raw_comments = fetch_comments("artificial", "2026-06-01", "2026-06-30")
comments_df = pd.DataFrame(raw_comments)

print(f"\nTOTAL COMMENTS: {len(raw_comments)}")
print("Comments shape:", comments_df.shape)

Comments collected: 100
Status 422, retrying... (1/5)
Status 422, retrying... (2/5)
Comments collected: 200
Status 422, retrying... (1/5)
Status 422, retrying... (2/5)
Comments collected: 300
Comments collected: 400
Comments collected: 500
Status 422, retrying... (1/5)
Comments collected: 600
Comments collected: 700
Status 422, retrying... (1/5)
Status 422, retrying... (2/5)
Comments collected: 800
Status 422, retrying... (1/5)
Status 422, retrying... (2/5)
Comments collected: 900
Comments collected: 1000
Status 422, retrying... (1/5)
Comments collected: 1100
Comments collected: 1200
Status 422, retrying... (1/5)
Status 422, retrying... (2/5)
Comments collected: 1300
Status 422, retrying... (1/5)
Status 422, retrying... (2/5)
Comments collected: 1400
Status 422, retrying... (1/5)
Status 422, retrying... (2/5)
Comments collected: 1500
Status 422, retrying... (1/5)
Comments collected: 1600
Status 422, retrying... (1/5)
Status 422, retrying... (2/5)
Comments collected: 1700
Status 422, re

In [9]:
import re

# --- 1. Combine post title + body into one "text" column ---
posts_df["text"] = posts_df["title"].fillna("") + " " + posts_df["selftext"].fillna("")
comments_df["text"] = comments_df["body"].fillna("")

# --- 2. Remove deleted/removed content ---
# These show up as literal placeholder strings when content is gone
deleted_markers = ["[deleted]", "[removed]"]

posts_clean = posts_df[~posts_df["text"].str.strip().isin(deleted_markers)].copy()
comments_clean = comments_df[~comments_df["text"].str.strip().isin(deleted_markers)].copy()

print(f"Posts after removing deleted/removed: {len(posts_clean)} (was {len(posts_df)})")
print(f"Comments after removing deleted/removed: {len(comments_clean)} (was {len(comments_df)})")

# --- 3. Remove bot comments ---
# AutoModerator is Reddit's most common bot; add more usernames here if you spot others later
bot_authors = ["AutoModerator"]

comments_clean = comments_clean[~comments_clean["author"].isin(bot_authors)].copy()
print(f"Comments after removing bots: {len(comments_clean)}")

# --- 4. Strip URLs and markdown formatting using regex ---
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)          # remove URLs
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)           # **bold** -> bold
    text = re.sub(r"\*(.*?)\*", r"\1", text)               # *italic* -> italic
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)        # [text](link) -> text
    text = re.sub(r"[#>`~]", "", text)                     # remove #, >, `, ~ symbols
    text = re.sub(r"\s+", " ", text).strip()               # collapse extra whitespace
    return text

posts_clean["clean_text"] = posts_clean["text"].apply(clean_text)
comments_clean["clean_text"] = comments_clean["text"].apply(clean_text)

# --- 5. Remove rows that are empty after cleaning ---
posts_clean = posts_clean[posts_clean["clean_text"].str.len() > 0]
comments_clean = comments_clean[comments_clean["clean_text"].str.len() > 0]

# --- 6. Deduplicate ---
posts_clean = posts_clean.drop_duplicates(subset="clean_text")
comments_clean = comments_clean.drop_duplicates(subset="clean_text")

print(f"\nFinal posts: {len(posts_clean)}")
print(f"Final comments: {len(comments_clean)}")

# --- 7. Preview ---
posts_clean[["clean_text"]].head()

Posts after removing deleted/removed: 2317 (was 2317)
Comments after removing deleted/removed: 18791 (was 20447)
Comments after removing bots: 18791

Final posts: 2264
Final comments: 18397


,clean_text
0,HeyGen for AI video side hustles: what it actu...
1,Why do AI agents keep repeating the same brows...
2,I think I broke AI He's been on the same exest...
3,What features do you think are most important ...
4,Ai slop security… [removed]


In [10]:
import spacy
from collections import Counter

# Load the small English model we installed back in Step 1
nlp = spacy.load("en_core_web_sm")

def tokenize(text):
    doc = nlp(text.lower())  # lowercase for consistency
    # keep only alphabetic tokens, remove stopwords (the, is, and...) and punctuation
    tokens = [token.text for token in doc if token.is_alpha and not token.is_stop]
    return tokens

# Apply to a sample first (spaCy is slower on 20k+ rows, sample keeps this fast for now)
sample_comments = comments_clean["clean_text"].sample(min(2000, len(comments_clean)), random_state=42)

all_tokens = []
for text in sample_comments:
    all_tokens.extend(tokenize(text))

print(f"Total tokens (from {len(sample_comments)} sampled comments): {len(all_tokens)}")

# --- Basic corpus stats ---
word_freq = Counter(all_tokens)
top_terms = word_freq.most_common(20)

print("\nTop 20 most common words:")
for word, count in top_terms:
    print(f"{word}: {count}")

Total tokens (from 2000 sampled comments): 43125

Top 20 most common words:
ai: 925
like: 380
people: 344
use: 282
think: 254
model: 230
time: 203
work: 192
actually: 190
way: 180
good: 172
real: 160
data: 160
need: 158
things: 150
models: 150
thing: 145
human: 140
know: 136
right: 125
